In [1]:
import sys

sys.setrecursionlimit(100000)

In [2]:
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.datasets import mnist
from tensorflow.keras.optimizers import Adam
import os
import warnings

warnings.filterwarnings('ignore')

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [3]:
from tensorflow import keras
from sklearn.model_selection import train_test_split

(x_all, y_all), _ = keras.datasets.mnist.load_data()

x_all = x_all.reshape((x_all.shape[0], -1)).astype("float32") / 255.0

x_temp, x_test, y_temp, y_test = train_test_split(x_all, y_all, test_size=0.2, random_state=42)

x_train, x_val, y_train, y_val = train_test_split(x_temp, y_temp, test_size=0.25, random_state=42)

In [4]:
(x_train, y_train), (x_val, y_val) = mnist.load_data()
x_train, x_val = x_train / 255.0, x_val / 255.0

print(f'Training data shape: {x_train.shape}')
print(f'Validation data shape: {x_val.shape}')

Training data shape: (60000, 28, 28)
Validation data shape: (10000, 28, 28)


In [5]:
def build_model(hp):
    model = Sequential([
        Flatten(input_shape=(28, 28)),
        Dense(units=hp.Int('units', min_value=32, max_value=512, step=32), activation='relu'),
        Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG')),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [6]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=2,
    directory='my_dir',
    project_name='intro_to_kt'
)

tuner.search_space_summary()

Search space summary
Default search space size: 2
units (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 512, 'step': 32, 'sampling': 'linear'}
learning_rate (Float)
{'default': 0.0001, 'conditions': [], 'min_value': 0.0001, 'max_value': 0.01, 'step': None, 'sampling': 'log'}


In [7]:
tuner.search(x_train, y_train, epochs=5, validation_data=(x_val, y_val))
tuner.results_summary()

Trial 10 Complete [00h 00m 29s]
val_accuracy: 0.9668000042438507

Best val_accuracy So Far: 0.980650007724762
Total elapsed time: 00h 06m 01s
Results summary
Results in my_dir\intro_to_kt
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 01 summary
Hyperparameters:
units: 224
learning_rate: 0.0015324287478969592
Score: 0.980650007724762

Trial 08 summary
Hyperparameters:
units: 352
learning_rate: 0.0011779860142648606
Score: 0.9802500009536743

Trial 00 summary
Hyperparameters:
units: 384
learning_rate: 0.0012250206925988468
Score: 0.9793500006198883

Trial 05 summary
Hyperparameters:
units: 192
learning_rate: 0.000886727494963954
Score: 0.9790500104427338

Trial 07 summary
Hyperparameters:
units: 224
learning_rate: 0.0004980947137951163
Score: 0.9775499999523163

Trial 04 summary
Hyperparameters:
units: 224
learning_rate: 0.00042399224034472463
Score: 0.9770999848842621

Trial 03 summary
Hyperparameters:
units: 128
learning_rate: 0.002757546352225043
Score:

In [8]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"""

The optimal number of units in the first dense layer is {best_hps.get('units')}.

The optimal learning rate for the optimizer is {best_hps.get('learning_rate')}.

""")

model = tuner.hypermodel.build(best_hps)
model.fit(x_train, y_train, epochs=10, validation_split=0.2)

test_loss, test_acc = model.evaluate(x_val, y_val)
print(f'Test accuracy: {test_acc}')

 

The optimal number of units in the first dense layer is 224. 

The optimal learning rate for the optimizer is 0.0015324287478969592. 


Epoch 1/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.8831 - loss: 0.3855 - val_accuracy: 0.9532 - val_loss: 0.1521
Epoch 2/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9707 - loss: 0.0982 - val_accuracy: 0.9722 - val_loss: 0.0921
Epoch 3/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9813 - loss: 0.0599 - val_accuracy: 0.9734 - val_loss: 0.0891
Epoch 4/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9851 - loss: 0.0447 - val_accuracy: 0.9761 - val_loss: 0.0829
Epoch 5/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9905 - loss: 0.0315 - val_accuracy: 0.9761 - val_loss: 0.0832
Epoch 6/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9923 - loss: 0.0231 - val_accuracy: 0.9763 - val_loss: 0.0906
Epoch 7/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9937 - loss: 